# Crawl và đo benchmark arXiv trên Colab CPU

Mỗi người chạy toàn bộ pipeline cho owner của mình. Chọn **Hardware accelerator: None (CPU)**, đổi `OWNER`, xác nhận Drive rồi **Run all** và cấp quyền mount.

Notebook tự chạy theo đợt 500 task mới → kiểm tra/thống kê → snapshot/verify → sao lưu lên Drive → resume. Sau khi đi hết dải, chỉ thử bổ sung lỗi retryable tối đa 2 lượt. Giữ 2 snapshot mới nhất và JSON của mọi lượt đo. Không cần đổi measurement sau từng đợt.

Corpus/SQLite ở ổ VM; Drive chung `/content/drive/MyDrive/arXiv-data` chỉ nhận bản bàn giao. Mỗi người cần quyền ghi đúng thư mục chung. VM bị thu hồi vẫn cần mở phiên và restore; chương trình không giữ VM sống hay sao lưu nền trong lúc runner đang chạy.

## 1. Điền cấu hình

Điền `COMMIT_SHA` là SHA 40 ký tự sau khi commit/push runner có `--retry-before-attempt`, `--stop-after-seconds` và `--references-rate-limit-threshold`. Owner: `trong`, `khang`, `hoang`, `man-nhi`, `huynh-nhi`. Mỗi người đổi `OWNER`; `RUN_ID` tự tạo riêng. `full` bỏ giới hạn mẫu; `pilot` dùng `MAX_CANDIDATES`/`MAX_PAPERS`. Không đổi run pilot thành full bằng resume.

`STOP_AFTER_TASKS=500` là số task mới mỗi đợt, không phải số bài. Discovery cả dải là một task nên có thể lâu. `MEASUREMENT_ID` chỉ là ID khởi đầu; vòng điều phối tự chọn ID mới. `AUTO_RETRY_PASSES=2` giới hạn lượt thử lại sau khi đi hết dải. Chạm `AUTO_MAX_BATCHES=200` là dừng an toàn, không có nghĩa đã crawl đủ.

`SNAPSHOT_MAX_FILES=500_000` dùng cho export/verify/restore, được ghi trong `run.json`. Dùng cùng giới hạn khi đọc snapshot; giới hạn byte vẫn là 2 GiB/file và 20 GiB tổng. 

Drive chung đã có. Thêm shortcut vào My Drive nếu cần; đường dẫn phải trỏ đúng thư mục đã chia sẻ. Mỗi owner/run chỉ có một phiên chạy. Không chạy notebook này song song với runner cũ cùng corpus.

Lượt đầu chỉ làm task chưa hoàn tất; lỗi được hoãn đến sau khi duyệt hết metadata/nguồn/references trong dải. Mỗi lỗi retryable được thử tối đa một lần trong mỗi lượt bổ sung. Mốc 500 task chỉ để sao lưu, không bắt đầu retry.

`STOP_AFTER_SECONDS=600` là mốc mềm giữa các task, tính từ sau phục hồi. Task đang chạy, kiểm tra và backup có thể làm tổng thời gian dài hơn 10 phút. `REFERENCES_RATE_LIMIT_THRESHOLD=2` dừng sau hai Result references mới liên tiếp có HTTP 429; chỉ chạy tiếp khi bạn chủ động chạy lại sau khi kiểm tra. `AUTO_BATCHES_PER_CALL=1` chạy thử một đợt, không đặt lại ngân sách tổng. Key nạp từ Colab Secrets `S2_API_KEY`; preflight HTTP 200 mới cho tạo batch mới.


In [ ]:
from pathlib import Path

COMMIT_SHA = ""  
PREVIOUS_PIPELINE_COMMIT = ""  
OWNER = "trong"
RUN_ID = f"m1-colab-{OWNER}-01"
RUN_MODE = "full"  # pilot | full | resume
MEASUREMENT_ID = "cold-01"  # chỉ là ID khởi đầu; auto tự tăng sau mỗi đợt
MAX_CANDIDATES = 40
MAX_PAPERS = 10
STOP_AFTER_TASKS = 500  # auto chạy từng đợt rồi sao lưu trước khi tiếp tục
STOP_AFTER_SECONDS = 600  # mốc mềm: kiểm tra giữa task, không cắt task đang chạy
REFERENCES_RATE_LIMIT_THRESHOLD = 2  # hai bài references mới liên tiếp có HTTP 429
REQUIRE_S2_KEY = True  # key nạp từ Colab Secrets, không ghi vào cấu hình
AUTO_BATCHES_PER_CALL = None  # đặt 1 để thử đúng một batch rồi sao lưu/dừng
AUTO_RETRY_PASSES = 2
AUTO_MAX_BATCHES = 200  # trần an toàn
RESTORED_AUTOMATION = None
WORK_ROOT = Path("/content/arxiv-work")
CORPUS_PATH = ""  # điền đường dẫn corpus đã restore khi tiếp tục ở vị trí khác
RESTORE_BUNDLE = ""  # đường dẫn zip trên Drive khi cần khôi phục trong phiên mới
TEAM_DRIVE_FOLDER_URL = "https://drive.google.com/drive/folders/1w9cVHms6Q9aakHKMSJY5DGzmBRAPleeT?usp=drive_link"
TEAM_DRIVE_DIR = "/content/drive/MyDrive/arXiv-data"  # đường dẫn dự phòng nếu thư mục nằm trực tiếp trong MyDrive
SNAPSHOT_MAX_FILES = 500_000  # dùng chung cho export/verify/restore
REPO_DIR = Path(f"/content/arxiv-code-{COMMIT_SHA[:12]}")


## 2. Lấy đúng code và cài dependencies

Cell này clone repo công khai nếu chưa có; không xóa checkout hoặc dữ liệu cũ. Nếu checkout không khớp commit hoặc code pipeline bị sửa, cell dừng để tránh gán sai provenance.

Nếu cần API key Semantic Scholar, đặt `S2_API_KEY` trong môi trường bằng cách quản lý secrets; không ghi key vào notebook hoặc in ra output.


In [ ]:
import importlib.metadata
import json
import os
import platform
import re
import subprocess
import sys

if not re.fullmatch(r"[0-9a-f]{40}", COMMIT_SHA):
    raise ValueError("Điền COMMIT_SHA 40 ký tự do cung cấp trước khi chạy.")
if OWNER not in {"trong", "khang", "hoang", "man-nhi", "huynh-nhi"}:
    raise ValueError("OWNER không thuộc nhóm.")
if RUN_MODE not in {"pilot", "full", "resume"}:
    raise ValueError("RUN_MODE phải là pilot, full hoặc resume.")
for name, value in {"RUN_ID": RUN_ID, "MEASUREMENT_ID": MEASUREMENT_ID}.items():
    if not isinstance(value, str) or not re.fullmatch(r"[A-Za-z0-9_-]+", value):
        raise ValueError(f"{name} chỉ gồm chữ, số, dấu gạch ngang và gạch dưới.")
if sys.version_info < (3, 10):
    raise RuntimeError("Pipeline cần Python >= 3.10.")
if WORK_ROOT.resolve().is_relative_to(Path("/content/drive").resolve()):
    raise ValueError("WORK_ROOT phải ở ổ làm việc local, không đặt SQLite trong Drive.")

REPO_URL = "https://github.com/ChickenDev-cyber/arxiv-data-science.git"
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--no-checkout", REPO_URL, str(REPO_DIR)], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--detach", COMMIT_SHA], check=True)

def git_output(*args):
    return subprocess.check_output(["git", "-C", str(REPO_DIR), *args], text=True).strip()

def ensure_release():
    if Path(git_output("rev-parse", "--show-toplevel")).resolve() != REPO_DIR.resolve():
        raise RuntimeError("REPO_DIR không phải gốc checkout đã chọn.")
    origin = git_output("remote", "get-url", "origin").removesuffix(".git").lower()
    if origin != REPO_URL.removesuffix(".git").lower():
        raise RuntimeError("Checkout không thuộc repo mã nguồn của nhóm.")
    head = git_output("rev-parse", "HEAD")
    if head != COMMIT_SHA or git_output("status", "--porcelain", "--", "src/arxiv_pipeline", "src/requirements.txt"):
        raise RuntimeError("Commit không khớp hoặc source pipeline đang có sửa đổi chưa commit.")
    return head

ensure_release()
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO_DIR / "src/requirements.txt")], check=True)
PIPELINE_ENV = {**os.environ, "PYTHONPATH": str(REPO_DIR / "src")}
schema = subprocess.check_output([sys.executable, "-c", "from arxiv_pipeline.contracts import SCHEMA_VERSION; print(SCHEMA_VERSION)"], cwd=REPO_DIR, env=PIPELINE_ENV, text=True).strip()
if schema != "1.2":
    raise RuntimeError(f"Cần schema 1.2, checkout hiện tại trả {schema}.")
print(json.dumps({"commit": COMMIT_SHA, "python": platform.python_version(), "os": platform.platform(), "cpu_count": os.cpu_count(), "psutil": importlib.metadata.version("psutil")}, indent=2))


### Nạp key Semantic Scholar
Trong Colab, mở biểu tượng chìa khóa (Secrets), thêm `S2_API_KEY` và bật quyền truy cập cho notebook. Không dán key vào ô cấu hình hoặc output. Ô này chỉ nạp key; auto-run sẽ thử một request trước khi tạo batch mới. Việc phục hồi backup đang dở vẫn chạy được khi chưa nạp key.


In [ ]:
# Không hiển thị giá trị key, kể cả khi có lỗi đọc Colab Secrets.
try:
    from google.colab import userdata
    _loaded_s2_key = userdata.get('S2_API_KEY')
except Exception:
    _loaded_s2_key = os.environ.get('S2_API_KEY')
if _loaded_s2_key:
    os.environ['S2_API_KEY'] = _loaded_s2_key
    PIPELINE_ENV['S2_API_KEY'] = _loaded_s2_key
    print('Đã nạp S2_API_KEY; giá trị được giữ kín.')
else:
    PIPELINE_ENV.pop('S2_API_KEY', None)
    print('Chưa có S2_API_KEY. Auto-run sẽ yêu cầu key trước batch dữ liệu thật mới.')
del _loaded_s2_key


In [ ]:
import hashlib
import shutil
import signal
import subprocess
import tempfile
import sys
from pathlib import Path


def run_cli(args, *, allowed=(0,), env=None):
    """Run argv without a shell, stream progress and preserve nonzero outcomes."""
    with subprocess.Popen(args, cwd=REPO_DIR, env=PIPELINE_ENV if env is None else env, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1,
                          start_new_session=True) as process:
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
            exit_code = process.wait()
        except KeyboardInterrupt:
            process.send_signal(signal.SIGINT)
            try:
                process.wait(timeout=45)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            raise
    if exit_code not in allowed:
        raise RuntimeError(f"Lệnh trả mã {exit_code}; đọc output trước khi tiếp tục.")
    return exit_code



def run_snapshot_cli(args):
    """Use the same configured file cap when exporting or reading snapshots."""
    if type(SNAPSHOT_MAX_FILES) is not int or SNAPSHOT_MAX_FILES < 1:
        raise ValueError("SNAPSHOT_MAX_FILES phải là số nguyên dương.")
    return run_cli([sys.executable, "-m", "arxiv_pipeline.snapshots", *args,
                    "--max-files", str(SNAPSHOT_MAX_FILES)])


def sha256_file(path):
    result = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            result.update(chunk)
    return result.hexdigest()


def copy_verified(source, destination, *, expected_sha256=None):
    """Copy to a temporary file, verify bytes, then publish; never overwrite a different archive."""
    source, destination = Path(source), Path(destination)
    expected = sha256_file(source) if expected_sha256 is None else expected_sha256
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists():
        if sha256_file(destination) == expected:
            return destination
        raise FileExistsError(f"Không ghi đè file khác nội dung: {destination}")
    with tempfile.NamedTemporaryFile(dir=destination.parent, prefix=destination.name + ".", suffix=".part", delete=False) as stream:
        temporary = Path(stream.name)
    try:
        shutil.copyfile(source, temporary)
        if sha256_file(temporary) != expected:
            raise ValueError("Bản sao không khớp SHA-256; chưa công bố file đích.")
        temporary.replace(destination)
    finally:
        temporary.unlink(missing_ok=True)
    return destination


def resolve_team_drive(folder_url, configured_path='', *, mount_root=Path('/content/drive')):
    """Find a mounted shared target; never create a replacement folder."""
    import re
    from urllib.parse import urlparse

    parsed = urlparse(folder_url)
    match = re.search(r'/folders/([A-Za-z0-9_-]+)(?:/|$)', parsed.path)
    if parsed.scheme != 'https' or parsed.hostname != 'drive.google.com' or match is None:
        raise ValueError('TEAM_DRIVE_FOLDER_URL phải là link thư mục Google Drive hợp lệ.')
    folder_id = match[1]
    root = Path(mount_root).resolve()

    def checked(path):
        path = Path(path).resolve()
        if not path.is_relative_to(root):
            raise ValueError('Thư mục bàn giao phải nằm trong Drive đã mount.')
        parts = path.relative_to(root).parts
        if parts and parts[0] == '.shortcut-targets-by-id':
            if len(parts) < 2 or parts[1] != folder_id:
                raise ValueError('Lối tắt trỏ đến ID khác link thư mục chung; kiểm tra TEAM_DRIVE_DIR.')
        return path

    # DriveFS can expose mounted shortcuts under the target ID even when the
    # shortcut's display name differs between accounts. This layout is optional.
    targets = root / '.shortcut-targets-by-id' / folder_id
    if targets.is_dir():
        candidates = [checked(p) for p in targets.iterdir() if p.is_dir()]
        if len(candidates) == 1:
            return candidates[0]
        if candidates:
            selected = checked(configured_path) if configured_path else None
            if selected in candidates:
                return selected
            raise ValueError('Có nhiều đường dẫn cho ID chung; điền TEAM_DRIVE_DIR bằng đường dẫn chính xác.')
    if configured_path:
        selected = checked(configured_path)
        if selected.is_dir():
            return selected
    raise FileNotFoundError(
        f'Chưa thấy thư mục chung trong Drive đã mount: {folder_url}\n'
        'Mở link bằng đúng tài khoản Colab, thêm lối tắt vào Drive của tôi (My Drive), '
        'rồi chạy lại cell mount. Nếu thư mục thuộc tài khoản này, sao chép đường dẫn '
        'thực tế từ bảng Files vào TEAM_DRIVE_DIR. Không tạo thư mục khác cùng tên.'
    )


In [ ]:
import json
import importlib.metadata
import os
import re
import shlex
import math
import time
import sys
from datetime import datetime, timezone


def automation_json(path, document):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.NamedTemporaryFile(mode='w', dir=path.parent, suffix='.part', delete=False) as stream:
        temporary = Path(stream.name)
        json.dump(document, stream, ensure_ascii=False, indent=2)
        stream.write('\n')
    try:
        os.replace(temporary, path)
    finally:
        temporary.unlink(missing_ok=True)


def measurement_order(name):
    match = re.fullmatch(r'(cold|resume)-(\d+)', name)
    if not match or int(match[2]) < 1:
        raise ValueError('Measurement phải có dạng cold-01 hoặc resume-01.')
    return (match[1] == 'resume', int(match[2]))


class NotebookBatchWorker:
    """Execute, inspect and durably hand off one measured pipeline invocation."""

    def __init__(self, *, repo_dir, env, work_root, output_root, drive_root,
                 owner, run_id, pipeline_commit, initial_args,
                 stop_after_tasks=500, max_files=500_000, expected_mock=False,
                 cli=None, restored_state=None, historical_pipeline_commit=None,
                 stop_after_seconds=None, references_rate_limit_threshold=None, require_s2_key=False):
        if drive_root is None or not Path(drive_root).is_dir():
            raise ValueError('Auto-run cần thư mục Drive đã mount và kiểm tra quyền ghi.')
        if owner not in {'trong', 'khang', 'hoang', 'man-nhi', 'huynh-nhi'}:
            raise ValueError('Owner không thuộc nhóm.')
        if not re.fullmatch(r'[A-Za-z0-9_-]+', run_id) or not re.fullmatch(r'[0-9a-f]{40}', pipeline_commit):
            raise ValueError('Run ID hoặc commit không hợp lệ.')
        for value in (stop_after_tasks, max_files):
            if type(value) is not int or value < 1:
                raise ValueError('Giới hạn task/file phải là số nguyên dương.')
        if stop_after_seconds is not None and (type(stop_after_seconds) not in (int, float)
                or not math.isfinite(stop_after_seconds) or stop_after_seconds <= 0):
            raise ValueError('stop_after_seconds phải hữu hạn và dương.')
        if references_rate_limit_threshold is not None and (
                type(references_rate_limit_threshold) is not int or references_rate_limit_threshold < 1):
            raise ValueError('references_rate_limit_threshold phải là số nguyên dương.')
        if type(require_s2_key) is not bool:
            raise ValueError('require_s2_key phải là bool.')
        self.stop_after_seconds = stop_after_seconds
        self.references_rate_limit_threshold = references_rate_limit_threshold
        self.require_s2_key = require_s2_key
        self._preflight_ready = False
        self.repo_dir, self.env = Path(repo_dir), dict(env)
        self.work_root, self.output_root, self.drive_root = Path(work_root), Path(output_root), Path(drive_root)
        self.owner, self.run_id, self.pipeline_commit = owner, run_id, pipeline_commit
        self.initial_args = list(initial_args)
        self.stop_after_tasks, self.max_files = stop_after_tasks, max_files
        self.expected_mock, self.cli = expected_mock, cli or run_cli
        self.restored_state = restored_state
        if historical_pipeline_commit and not re.fullmatch(r'[0-9a-f]{40}', historical_pipeline_commit):
            raise ValueError('Previous pipeline commit phải là SHA 40 ký tự.')
        self.historical_pipeline_commit = historical_pipeline_commit

    def _capture(self, arguments, allowed=(0,)):
        result = subprocess.run(arguments, cwd=self.repo_dir, env=self.env, text=True, capture_output=True)
        if result.returncode not in allowed:
            raise RuntimeError(result.stderr or result.stdout or f'Lệnh trả mã {result.returncode}')
        return json.loads(result.stdout)

    def _snapshot(self, arguments):
        if type(self.max_files) is not int or self.max_files < 1:
            raise ValueError('Giới hạn snapshot phải là số nguyên dương.')
        return self.cli([sys.executable, '-m', 'arxiv_pipeline.snapshots', *arguments,
                         '--max-files', str(self.max_files)])

    def _identity(self, document, *, historical=False):
        commits = {self.pipeline_commit}
        if historical and self.historical_pipeline_commit:
            commits.add(self.historical_pipeline_commit)
        if (document.get('owner') != self.owner or document.get('run_id') != self.run_id
                or document.get('pipeline_commit') not in commits):
            raise ValueError('Metadata không khớp owner/run ID/commit của phiên này.')

    def make_batch(self, mode, measurement_id, *, retry_before_attempt=0):
        if mode not in ('pilot', 'full', 'resume'):
            raise ValueError('Mode không hợp lệ.')
        measurement_order(measurement_id)
        _validate_nonnegative_int(retry_before_attempt, 'retry_before_attempt')
        root = self.work_root / 'measurements' / self.run_id / measurement_id
        runner = [sys.executable, '-m', 'arxiv_pipeline.runner', '--run-id', self.run_id,
                  '--output-root', str(self.output_root)]
        runner += ['--resume'] if mode == 'resume' else self.initial_args
        runner += ['--stop-after-tasks', str(self.stop_after_tasks),
                   '--retry-before-attempt', str(retry_before_attempt)]
        if self.stop_after_seconds is not None:
            runner += ['--stop-after-seconds', str(self.stop_after_seconds)]
        if self.references_rate_limit_threshold is not None:
            runner += ['--references-rate-limit-threshold', str(self.references_rate_limit_threshold)]
        command = [sys.executable, '-m', 'arxiv_pipeline.benchmark', '--run-id', self.run_id,
                   '--output-root', str(self.output_root), '--output-dir', str(root / 'benchmark'),
                   '--cache-mode', 'mixed' if mode == 'resume' else 'cold', '--cmd', shlex.join(runner)]
        return {'owner': self.owner, 'run_id': self.run_id, 'pipeline_commit': self.pipeline_commit,
                'mode': mode, 'measurement_id': measurement_id, 'output_root': str(self.output_root),
                'measurement_root': str(root), 'benchmark_dir': str(root / 'benchmark'),
                'statistics_dir': str(root / 'statistics'),
                'handoff_dir': str(self.work_root / 'handoffs' / self.run_id / measurement_id),
                'cache_mode': 'mixed' if mode == 'resume' else 'cold', 'command': command,
                'interrupted': False, 'retry_before_attempt': retry_before_attempt}

    def _validate_batch(self, batch):
        self._identity(batch)
        expected = self.make_batch(batch['mode'], batch['measurement_id'],
                                   retry_before_attempt=batch.get('retry_before_attempt', 0))
        for name in ('output_root', 'measurement_root', 'benchmark_dir', 'statistics_dir', 'handoff_dir', 'cache_mode'):
            if batch.get(name) != expected[name]:
                raise ValueError(f'Đường dẫn/cấu hình batch không khớp: {name}')

    def preflight(self):
        """Check capabilities and authenticated access before journaling fresh work."""
        self._preflight_ready = False
        if not getattr(self, '_checked_pass_support', False):
            help_result = subprocess.run([sys.executable, '-m', 'arxiv_pipeline.runner', '--help'],
                cwd=self.repo_dir, env=self.env, text=True, capture_output=True)
            required = ['--retry-before-attempt']
            if self.stop_after_seconds is not None:
                required.append('--stop-after-seconds')
            if self.references_rate_limit_threshold is not None:
                required.append('--references-rate-limit-threshold')
            if help_result.returncode != 0 or any(flag not in help_result.stdout for flag in required):
                raise RuntimeError('COMMIT_SHA chưa hỗ trợ các điều khiển chạy tiếp. '
                                   'Commit/push runner mới rồi ghim SHA đó trước khi crawl.')
            self._checked_pass_support = True
        if self.require_s2_key and not self.expected_mock:
            if not self.env.get('S2_API_KEY', '').strip():
                raise ValueError('Chưa có S2_API_KEY. Thêm Colab Secret và chạy lại ô nạp key.')
            # Assigned range's first paper; no extra cross-owner search request.
            probe_ids = {'trong': '2504.17523', 'khang': '2506.20788', 'hoang': '2503.06914',
                         'man-nhi': '2501.12642', 'huynh-nhi': '2409.03594'}
            probe = self._capture([sys.executable, '-m', 'arxiv_pipeline.references.preflight',
                '--arxiv-id', probe_ids[self.owner], '--output-root', str(self.output_root),
                '--run-id', self.run_id], allowed=(0, 2))
            if probe.get('ok') is not True or probe.get('http_status') != 200:
                raise RuntimeError('S2 preflight chưa đạt: ' + json.dumps(probe, ensure_ascii=False)
                    + '. Chưa tạo batch mới; kiểm tra quyền key hoặc chờ cooldown rồi chạy lại.')
            print('S2 preflight: HTTP 200; key đã được nạp (không hiển thị key).')
            time.sleep(1.1)  # Separate this probe from the next measured S2 request.
        self._preflight_ready = True

    def execute_batch(self, batch):
        if not self._preflight_ready:
            self.preflight()
        self._preflight_ready = False
        self._validate_batch(batch)
        self._assert_current_corpus()
        measurement = Path(batch['measurement_root'])
        if Path(batch['handoff_dir']).exists():
            raise FileExistsError('Measurement đã có gói bàn giao; không chạy lại vào gói cũ.')
        if measurement.exists() and any(measurement.iterdir()):
            raise FileExistsError('Không ghi đè measurement đã có dữ liệu.')
        if batch['mode'] != 'resume' and self.output_root.exists() and any(self.output_root.iterdir()):
            raise FileExistsError('Corpus đã có; cần resume thay vì chạy cold lại.')
        measurement.mkdir(parents=True, exist_ok=True)
        automation_json(measurement / 'invocation.json', batch)
        try:
            result = self.cli(batch['command'], allowed=(0, 2, 130), env=self.env)
            batch['interrupted'] = result == 130
        except KeyboardInterrupt:
            batch['interrupted'] = True
            if not self.batch_completed(batch):
                raise
        state = self._run_state()
        batch['core_revision'] = [state['updated_at'], state['attempt_count']]
        automation_json(measurement / 'invocation.json', batch)
        return batch

    def batch_completed(self, batch):
        return (Path(batch['benchmark_dir']) / f'{self.run_id}.json').is_file()

    def _run_state(self, retry_before_attempt=0):
        # Inspect failed/latest results through the pinned StateStore implementation.
        code = '''import json,sys
from pathlib import Path
from arxiv_pipeline.runner.state import StateStore
from arxiv_pipeline.runner.service import _reusable
with StateStore.readonly(Path(sys.argv[1])/"state.sqlite",sys.argv[2]) as store:
 record=store.read_run_record()
 boundary=int(sys.argv[3])
 max_attempt_id=store.connection.execute("SELECT coalesce(max(attempt_id),0) FROM attempts WHERE run_id=?",(sys.argv[2],)).fetchone()[0]
 processed=store.connection.execute("SELECT count(DISTINCT task_key) FROM attempts WHERE run_id=? AND attempt_id>? AND result_valid=1",(sys.argv[2],boundary)).fetchone()[0]
 retry=[]
 raw_count=version_count=discovery_count=0
 for row in store.connection.execute("SELECT t.task_key,a.result_json,a.result_valid FROM run_tasks t LEFT JOIN attempts a ON a.attempt_id=t.latest_attempt_id WHERE t.run_id=?",(sys.argv[2],)):
  task=store.get_task(row["task_key"])
  accepted=task["accepted_result"]
  if accepted and accepted.get("data"):
   data=accepted["data"]
   if accepted["step"]=="references": raw_count+=len(data.get("raw_references",[]))
   if accepted["step"]=="metadata": version_count+=len(data.get("versions",[]))
   if accepted["step"]=="discovery": discovery_count+=len(data.get("papers",[]))+len(data.get("excluded_ids",[]))
  if accepted is not None and _reusable(accepted): continue
  if row["result_json"] is None or row["result_valid"]!=1: continue
  result=json.loads(row["result_json"])
  if result.get("error") and result["error"].get("retryable") is True:
   retry.append({k:result.get(k) for k in ("step","arxiv_id","version","error")})
 print(json.dumps({"run_state":record.status,"max_attempt_id":max_attempt_id,"processed_in_pass":processed,"config":record.config,"retryable":retry,"raw_reference_entries":raw_count,"metadata_version_entries":version_count,"discovery_classified":discovery_count,"updated_at":record.updated_at,"attempt_count":store.connection.execute("SELECT count(*) FROM attempts WHERE run_id=?",(sys.argv[2],)).fetchone()[0]}))
'''
        state = self._capture([sys.executable, '-c', code, str(self.output_root), self.run_id, str(retry_before_attempt)])
        config = state['config']
        if (config['run_id'] != self.run_id or config['owners'] != [self.owner]
                or config['is_mock'] is not self.expected_mock
                or Path(config['output_root']).resolve() != self.output_root.resolve()):
            raise ValueError('Corpus không khớp owner/run ID/mode mock.')
        if state['run_state'] not in ('paused', 'finished'):
            raise ValueError('Corpus chưa dừng; không chạy thêm hoặc sao lưu khi runner còn hoạt động.')
        execution = self.output_root / 'runs' / self.run_id / 'execution.json'
        state['pause_reason'] = None
        if execution.exists():
            saved = json.loads(execution.read_text())
            if (saved.get('format_version') != 1 or saved.get('run_id') != self.run_id
                    or saved.get('run_state') != state['run_state']
                    or saved.get('max_attempt_id') != state['max_attempt_id']
                    or saved.get('pause_reason') not in (None, 'task_limit', 'time_limit',
                                                        'interrupted', 'references_rate_limited')):
                raise ValueError('execution.json không khớp trạng thái runner; không dùng lý do dừng cũ.')
            state['pause_reason'] = saved['pause_reason']
        return state

    def _assert_revision(self, batch, report=None):
        revision = batch.get('core_revision')
        invocation = Path(batch['measurement_root']) / 'invocation.json'
        if revision is None and invocation.is_file():
            record = json.loads(invocation.read_text())
            self._identity(record)
            revision = record.get('core_revision')
        if revision is None and report is not None:
            revision = report.get('core_revision')
        state = self._run_state(batch.get('retry_before_attempt', 0))
        if revision is not None and revision != [state['updated_at'], state['attempt_count']]:
            raise ValueError('Corpus đã tiến sang lượt khác; không sửa số đo hoặc sao lưu measurement cũ.')
        return state

    def inspect_batch(self, batch):
        self._validate_batch(batch)
        state = self._assert_revision(batch)
        inspected = self._capture([sys.executable, '-m', 'arxiv_pipeline.collection', 'inspect',
            '--output-root', str(self.output_root), '--run-id', self.run_id], allowed=(0, 2))
        with tempfile.TemporaryDirectory(prefix='arxiv-batch-statistics-') as temporary:
            self.cli([sys.executable, '-m', 'arxiv_pipeline.statistics', '--output-root', str(self.output_root),
                      '--run-id', self.run_id, '--output-dir', temporary])
            stats = json.loads((Path(temporary) / f'{self.run_id}.json').read_text())
        bench = json.loads((Path(batch['benchmark_dir']) / f'{self.run_id}.json').read_text())
        if (stats['scope'] != bench['scope'] or stats['is_mock'] is not self.expected_mock
                or bench['is_mock'] is not self.expected_mock or bench['cache_mode'] != batch['cache_mode']):
            raise ValueError('Benchmark và corpus không cùng phạm vi/mode.')
        target = Path(batch['statistics_dir']) / f'{self.run_id}.json'
        if target.exists():
            if json.loads(target.read_text()) != stats:
                raise ValueError('Thống kê cũ khác corpus hiện tại; không ghi đè measurement.')
        else:
            automation_json(target, stats)
        progress_fields = ('papers_discovered', 'metadata_complete_papers', 'source_versions_success',
                           'source_versions_unsupported', 'references_fully_fetched_papers', 'references_complete_total')
        progress = [stats['summary'][name] for name in progress_fields]
        progress += [batch.get('retry_before_attempt', 0), state['processed_in_pass']]
        progress += [inspected['accepted_results'], stats['scope']['discovery_complete'], stats['scope']['versions_complete'],
                     state['raw_reference_entries'], state['metadata_version_entries'], state['discovery_classified']]
        return {'run_state': state['run_state'], 'pause_reason': state['pause_reason'],
                'max_attempt_id': state['max_attempt_id'],
                'processed_in_pass': state['processed_in_pass'], 'core_revision': [state['updated_at'], state['attempt_count']], 'retryable_tasks': len(state['retryable']),
                'retryable': state['retryable'], 'progress_key': progress, 'summary': stats['summary'],
                'issues': inspected['issues'], 'stats': stats, 'bench': bench,
                'scope_complete': stats['scope']['discovery_complete'] and stats['scope']['versions_complete']}

    def _bundle_paths(self, batch):
        handoff = Path(batch['handoff_dir'])
        bundle = handoff.parent / f"{self.run_id}-{batch['measurement_id']}.zip"
        return handoff, bundle, bundle.with_suffix('.zip.sha256')

    def backup_batch(self, batch, report, continuation):
        self._validate_batch(batch)
        self._assert_revision(batch, report)
        latest_remote = self._latest_remote()
        if latest_remote is not None and measurement_order(latest_remote) > measurement_order(batch['measurement_id']):
            raise ValueError('Drive có measurement mới hơn; không bàn giao lại corpus cũ.')
        handoff, bundle, checksum = self._bundle_paths(batch)
        handoff.parent.mkdir(parents=True, exist_ok=True)
        if not handoff.exists():
            # Publish the directory only after its snapshot and metadata are complete.
            with tempfile.TemporaryDirectory(dir=handoff.parent, prefix='.building-') as temporary:
                staged = Path(temporary) / 'handoff'
                staged.mkdir()
                snapshot = staged / 'snapshot.tar.gz'
                self._snapshot(['export', '--output-root', str(self.output_root), '--run-id', self.run_id,
                    '--destination', str(snapshot), '--snapshot-id', f"{self.run_id}-{batch['measurement_id']}",
                    '--exporter-commit', self.pipeline_commit])
                self._snapshot(['verify', '--archive', str(snapshot)])
                for key, name in (('benchmark_dir', 'benchmark'), ('statistics_dir', 'statistics')):
                    shutil.copytree(batch[key], staged / name)
                metadata = {'owner': self.owner, 'run_id': self.run_id, 'pipeline_commit': self.pipeline_commit,
                    'measurement_id': batch['measurement_id'], 'schema_version': report['stats']['schema_version'],
                    'is_mock': report['bench']['is_mock'], 'scope': report['stats']['scope'],
                    'command': batch['command'], 'cache_mode': batch['cache_mode'],
                    'snapshot_limits': {'max_files': self.max_files}, 'automation': continuation,
                    'created_at': datetime.now(timezone.utc).isoformat(),
                    'environment': report['bench']['environment'], 'psutil': importlib.metadata.version('psutil'),
                    'retention': {'snapshots': 2, 'all_measurement_json': True}, 'interrupted': batch.get('interrupted', False)}
                automation_json(staged / 'run.json', metadata)
                staged.rename(handoff)
        metadata = json.loads((handoff / 'run.json').read_text())
        self._identity(metadata)
        if (metadata['measurement_id'] != batch['measurement_id'] or metadata['scope'] != report['stats']['scope']
                or metadata['is_mock'] is not self.expected_mock):
            raise ValueError('Gói đã có không khớp batch; giữ nguyên để kiểm tra.')
        if not bundle.exists():
            import zipfile
            with tempfile.NamedTemporaryFile(dir=handoff.parent, suffix='.zip.part', delete=False) as stream:
                temporary = Path(stream.name)
            try:
                with zipfile.ZipFile(temporary, 'w', compression=zipfile.ZIP_STORED) as archive:
                    for path in sorted(handoff.rglob('*')):
                        if path.is_file(): archive.write(path, path.relative_to(handoff))
                expected = sha256_file(temporary)
                if checksum.exists() and checksum.read_text().split()[0] != expected:
                    raise ValueError('Checksum đã có khác nội dung; không ghi đè.')
                if not checksum.exists(): checksum.write_text(f'{expected}  {bundle.name}\n')
                os.link(temporary, bundle)
            finally:
                temporary.unlink(missing_ok=True)
        expected = checksum.read_text().split()[0]
        if not re.fullmatch(r'[0-9a-f]{64}', expected) or sha256_file(bundle) != expected:
            raise ValueError('Gói local không khớp checksum; dừng trước batch tiếp theo.')
        destination = self.drive_root / self.owner / self.run_id / batch['measurement_id']
        copy_verified(bundle, destination / bundle.name, expected_sha256=expected)
        copy_verified(checksum, destination / checksum.name)
        # Preserve all small measurement files independently of the large snapshots.
        for path in [handoff / 'run.json', *sorted((handoff / 'benchmark').glob('*.json')),
                     *sorted((handoff / 'statistics').glob('*.json'))]:
            copy_verified(path, destination / path.relative_to(handoff))
        self._prune_snapshots()
        print('Đã chép và đối chiếu SHA-256 tại:', destination, flush=True)

    def _remote_measurements(self):
        directory = self.drive_root / self.owner / self.run_id
        if not directory.exists(): return []
        result = []
        for folder in directory.iterdir():
            if not folder.is_dir(): continue
            try: measurement_order(folder.name)
            except ValueError: continue
            bundle = folder / f'{self.run_id}-{folder.name}.zip'
            if bundle.is_file() and bundle.with_suffix('.zip.sha256').is_file():
                result.append(folder.name)
        return sorted(result, key=measurement_order)

    def _latest_remote(self):
        names = self._remote_measurements()
        return names[-1] if names else None

    def _assert_current_corpus(self):
        remote = self._latest_remote()
        if remote is None: return
        local_directory = self.work_root / 'measurements' / self.run_id
        completed = []
        if local_directory.exists():
            for folder in local_directory.iterdir():
                if not folder.is_dir(): continue
                try: measurement_order(folder.name)
                except ValueError: continue
                if (folder / 'benchmark' / f'{self.run_id}.json').is_file():
                    completed.append(folder.name)
        manifest = self.output_root.parent / 'manifest.json'
        if manifest.is_file():
            snapshot_id = json.loads(manifest.read_text()).get('snapshot_id', '')
            prefix = self.run_id + '-'
            if snapshot_id.startswith(prefix):
                candidate = snapshot_id[len(prefix):]
                measurement_order(candidate)
                completed.append(candidate)
        if not completed or max(map(measurement_order, completed)) < measurement_order(remote):
            raise ValueError('Drive có gói mới hơn corpus local (stale). Dừng và restore snapshot mới nhất trong phiên mới.')

    def _preserve_legacy_metrics(self, folder, bundle):
        import zipfile
        with zipfile.ZipFile(bundle) as archive:
            meta = json.loads(archive.read('run.json'))
            self._identity(meta, historical=True)
            if meta['measurement_id'] != folder.name:
                raise ValueError('Gói cũ không khớp measurement; không tự dọn.')
            names = ['run.json', f'benchmark/{self.run_id}.json',
                     f'benchmark/{self.run_id}-timing.json', f'statistics/{self.run_id}.json']
            with tempfile.TemporaryDirectory() as temporary:
                for name in names:
                    content = archive.read(name)  # Read whitelisted JSON only, never extract arbitrary paths.
                    json.loads(content)
                    source = Path(temporary) / name
                    source.parent.mkdir(parents=True, exist_ok=True)
                    source.write_bytes(content)
                    copy_verified(source, folder / name)

    def _prune_snapshots(self):
        # User policy: two newest full snapshots; every measurement JSON is retained.
        names = self._remote_measurements()
        parent = self.drive_root / self.owner / self.run_id
        # Both retained backups must be readable and match their checksums before
        # removing an older recovery point, including legacy handoffs.
        for name in names[-2:]:
            folder = parent / name
            bundle = folder / f'{self.run_id}-{name}.zip'
            expected = bundle.with_suffix('.zip.sha256').read_text().split()[0]
            if not re.fullmatch(r'[0-9a-f]{64}', expected) or sha256_file(bundle) != expected:
                raise ValueError('Snapshot giữ lại không khớp checksum; không dọn bản cũ.')
            self._preserve_legacy_metrics(folder, bundle)
        for name in names[:-2]:
            folder = parent / name
            bundle = folder / f'{self.run_id}-{name}.zip'
            checksum = bundle.with_suffix('.zip.sha256')
            expected = checksum.read_text().split()[0]
            if not re.fullmatch(r'[0-9a-f]{64}', expected) or sha256_file(bundle) != expected:
                raise ValueError('Gói cũ không khớp checksum; giữ nguyên, không tự dọn.')
            self._preserve_legacy_metrics(folder, bundle)
            # New snapshots and their checksums are already verified before reaching here.
            bundle.unlink()
            checksum.unlink()
        local_parent = self.work_root / 'handoffs' / self.run_id
        local_names = []
        if local_parent.exists():
            for folder in local_parent.iterdir():
                if not folder.is_dir() or not (folder / 'run.json').is_file(): continue
                try: measurement_order(folder.name)
                except ValueError: continue
                self._identity(json.loads((folder / 'run.json').read_text()), historical=True)
                local_names.append(folder.name)
        keep = set(sorted(local_names, key=measurement_order)[-2:])
        for name in local_names:
            if name in keep: continue
            # Only generated local archive copies are removed; corpus and JSON stay untouched.
            bundle = local_parent / f'{self.run_id}-{name}.zip'
            bundle.unlink(missing_ok=True)
            bundle.with_suffix('.zip.sha256').unlink(missing_ok=True)
            (local_parent / name / 'snapshot.tar.gz').unlink(missing_ok=True)

    def measurement_exists(self, measurement):
        measurement_order(measurement)
        local = self.work_root / 'measurements' / self.run_id / measurement
        remote = self.drive_root / self.owner / self.run_id / measurement
        handoff = self.work_root / 'handoffs' / self.run_id / measurement
        bundle = handoff.parent / f'{self.run_id}-{measurement}.zip'
        return local.exists() or remote.exists() or handoff.exists() or bundle.exists()

    def adopt_existing(self):
        if not (self.output_root / 'state.sqlite').exists():
            remote = self.drive_root / self.owner / self.run_id
            if remote.exists() and (any(remote.glob('*/*.zip.sha256')) or any(remote.glob('*/run.json'))):
                raise ValueError('Drive đã có run này; cần khôi phục (restore) snapshot trước, không chạy cold lại.')
            return None
        state = self._run_state()
        self._assert_current_corpus()
        if '--full-range' in self.initial_args and (state['config']['max_candidates'] is not None or state['config']['max_papers'] is not None):
            raise ValueError('Không biến run pilot thành toàn dải; chọn đúng run ID.')
        if self.restored_state is not None:
            self._identity(self.restored_state)
            return dict(self.restored_state)
        directory = self.work_root / 'measurements' / self.run_id
        measurements = sorted((p.name for p in directory.iterdir()
                               if p.is_dir() and re.fullmatch(r'(cold|resume)-[0-9]+', p.name)
                               and measurement_order(p.name)), key=measurement_order) if directory.exists() else []
        if not measurements:
            ordinal = 1
            while self.measurement_exists(f'resume-{ordinal:02d}'):
                ordinal += 1
            adopted = {'mode': 'resume', 'next_measurement': f'resume-{ordinal:02d}'}
            if state['run_state'] == 'finished':
                if state['retryable']:
                    adopted['retry_passes'] = 1
                    adopted['retry_before_attempt'] = state['max_attempt_id']
                else:
                    inspected = self._capture([sys.executable, '-m', 'arxiv_pipeline.collection', 'inspect',
                        '--output-root', str(self.output_root), '--run-id', self.run_id], allowed=(0, 2))
                    scope = json.loads((self.output_root / 'runs' / self.run_id / 'scope.json').read_text())['scope']
                    complete = scope['discovery_complete'] and scope['versions_complete']
                    with tempfile.TemporaryDirectory(prefix='arxiv-adopt-statistics-') as temporary:
                        self.cli([sys.executable, '-m', 'arxiv_pipeline.statistics', '--output-root', str(self.output_root),
                                  '--run-id', self.run_id, '--output-dir', temporary])
                        adopted['summary'] = json.loads((Path(temporary) / f'{self.run_id}.json').read_text())['summary']
                    adopted.update(terminal=True, stop_reason='finished' if complete and not inspected['issues']
                                   else 'finished_with_issues')
            return adopted
        latest = measurements[-1]
        handoff = self.work_root / 'handoffs' / self.run_id / latest
        metadata_path = handoff / 'run.json'
        if not metadata_path.is_file():
            raise ValueError('Lượt manual trước chưa xuất gói. Hoàn tất kiểm tra/sao lưu lượt đó trước khi bật auto.')
        metadata = json.loads(metadata_path.read_text())
        self._identity(metadata)
        mode = 'resume' if metadata['cache_mode'] == 'mixed' else ('full' if '--full-range' in metadata['command'][-1] else 'pilot')
        batch = self.make_batch(mode, latest)
        batch['command'] = metadata['command']  # Provenance only; this measured batch is never executed again.
        report = self.inspect_batch(batch)
        self.backup_batch(batch, report, metadata.get('automation', {}))
        if metadata.get('automation'):
            return metadata['automation']
        ordinal = 1
        while self.measurement_exists(f'resume-{ordinal:02d}'):
            ordinal += 1
        next_measurement = f'resume-{ordinal:02d}'
        if report['run_state'] == 'finished' and not report['retryable_tasks']:
            return {'mode': 'resume', 'next_measurement': next_measurement, 'terminal': True,
                    'stop_reason': 'finished' if not report['issues'] and report['scope_complete'] else 'finished_with_issues'}
        return {'mode': 'resume', 'next_measurement': next_measurement, 'previous_progress': report['progress_key'],
                'retry_before_attempt': report['max_attempt_id'] if report['run_state'] == 'finished' else 0,
                'retry_passes': 1 if report['run_state'] == 'finished' and report['retryable_tasks'] else 0}


"""controller.py - AutoCrawlSession controller for arXiv Colab automation.
Pure stdlib controller class implementing automated batch500->inspect->snapshot+Drive->resume.
"""

import fcntl
import json
import os
import re
import tempfile
from pathlib import Path
from typing import Any, Dict, Optional


def _validate_mode(mode: Any) -> str:
    if not isinstance(mode, str) or mode not in ("full", "pilot", "resume"):
        raise ValueError(f"Invalid mode: {mode!r} (expected 'full', 'pilot', or 'resume')")
    return mode


def _validate_measurement_id(mid: Any) -> str:
    if not isinstance(mid, str):
        raise ValueError(f"Invalid measurement ID: {mid!r}")
    m = re.match(r"^(cold|resume)-(\d+)$", mid)
    if not m:
        raise ValueError(f"Invalid measurement ID format: {mid!r} (expected cold-N or resume-N)")
    ordinal = int(m.group(2))
    if ordinal <= 0:
        raise ValueError(f"Measurement ordinal must be positive integer, got {mid!r}")
    return mid


def _validate_nonnegative_int(val: Any, name: str) -> int:
    if type(val) is not int or val < 0:
        raise ValueError(f"{name} must be a non-negative integer (not bool), got {val!r}")
    return val


def _validate_positive_int(val: Any, name: str) -> int:
    if type(val) is not int or val <= 0:
        raise ValueError(f"{name} must be a positive integer (not bool), got {val!r}")
    return val


def _validate_bool(val: Any, name: str) -> bool:
    if type(val) is not bool:
        raise ValueError(f"{name} must be a boolean, got {val!r}")
    return val


def _validate_pending(pending: Any) -> Optional[Dict[str, Any]]:
    if pending is None:
        return None
    if not isinstance(pending, dict):
        raise ValueError(f"pending must be a dict or None, got {pending!r}")
    if "batch" not in pending or not isinstance(pending["batch"], dict):
        raise ValueError(f"pending must contain a 'batch' dict, got {pending.get('batch')!r}")
    batch = pending["batch"]
    if "mode" not in batch or not isinstance(batch["mode"], str) or batch["mode"] not in ("full", "pilot", "resume"):
        raise ValueError(f"pending batch has invalid mode: {batch.get('mode')!r}")
    if "measurement_id" not in batch:
        raise ValueError("pending batch missing measurement_id")
    _validate_measurement_id(batch["measurement_id"])

    if "executed" not in pending or type(pending["executed"]) is not bool:
        raise ValueError(f"pending 'executed' must be a boolean, got {pending.get('executed')!r}")

    if "after" in pending and pending["after"] is not None:
        if not isinstance(pending["after"], dict):
            raise ValueError(f"pending 'after' must be a dict or None, got {pending['after']!r}")
    return pending


class AutoCrawlSession:
    """Session controller for automated arXiv crawling and benchmark execution."""

    def __init__(
        self,
        backend,
        *,
        owner: str,
        run_id: str,
        pipeline_commit: str,
        journal_path,
        initial_mode: str = "full",
        initial_measurement: str = "cold-01",
        max_retry_passes: int = 2,
        max_stall_retries: int = 2,
        max_batches: int = 200,
        resume_state: Optional[Dict[str, Any]] = None,
    ):
        # Validate limits (positive integers, reject bool)
        self.max_retry_passes = _validate_positive_int(max_retry_passes, "max_retry_passes")
        self.max_stall_retries = _validate_positive_int(max_stall_retries, "max_stall_retries")
        self.max_batches = _validate_positive_int(max_batches, "max_batches")

        # Validate initial mode and measurement ID
        self.mode = _validate_mode(initial_mode)
        self.next_measurement = _validate_measurement_id(initial_measurement)

        # Validate required identity fields
        if not owner or not isinstance(owner, str):
            raise ValueError("owner must be a non-empty string")
        if not run_id or not isinstance(run_id, str):
            raise ValueError("run_id must be a non-empty string")
        if not pipeline_commit or not isinstance(pipeline_commit, str):
            raise ValueError("pipeline_commit must be a non-empty string")

        self.backend = backend
        self.owner = owner
        self.run_id = run_id
        self.pipeline_commit = pipeline_commit
        self.journal_path = Path(journal_path)
        self.lock_path = self.journal_path.with_suffix(".lock")
        self.resume_state = resume_state

        self._lock_file = None

        # Default state attributes
        self.retry_passes = 0
        self.retry_before_attempt = 0
        self.stall_retries = 0
        self.previous_progress = None
        self.batches = 0
        self.terminal = False
        self.stop_reason = None
        self.pending = None
        self.last_summary = {}
        self.last_report = None

        # Validate resume_state identity and basic schema upfront if provided
        if self.resume_state is not None:
            self._validate_state_identity(self.resume_state, source="resume_state")
            self._validate_state_fields(self.resume_state)

        # Read-only check and identity validation of existing journal without disk writes or backend calls
        if self.journal_path.exists():
            self._load_journal_read_only()

    def _validate_state_identity(self, data: Dict[str, Any], source: str):
        if (
            data.get("owner") != self.owner
            or data.get("run_id") != self.run_id
            or data.get("pipeline_commit") != self.pipeline_commit
        ):
            raise ValueError(
                f"{source} identity mismatch: expected ({self.owner}, {self.run_id}, {self.pipeline_commit}) "
                f"but found ({data.get('owner')}, {data.get('run_id')}, {data.get('pipeline_commit')})"
            )

    def _validate_state_fields(self, data: Dict[str, Any]):
        if "mode" in data:
            _validate_mode(data["mode"])
        if "next_measurement" in data:
            _validate_measurement_id(data["next_measurement"])
        if "batches" in data:
            _validate_nonnegative_int(data["batches"], "batches")
        if "retry_before_attempt" in data:
            _validate_nonnegative_int(data["retry_before_attempt"], "retry_before_attempt")
        if "retry_passes" in data:
            _validate_nonnegative_int(data["retry_passes"], "retry_passes")
        if "stall_retries" in data:
            _validate_nonnegative_int(data["stall_retries"], "stall_retries")
        if "terminal" in data:
            _validate_bool(data["terminal"], "terminal")
        if "pending" in data:
            _validate_pending(data["pending"])

    def _load_journal_read_only(self):
        """Read-only load of journal from disk without modifying journal or calling backend."""
        with open(self.journal_path, "r", encoding="utf-8") as f:
            data = json.load(f)

        self._validate_state_identity(data, source="Journal")
        self._validate_state_fields(data)

        if "mode" in data and data["mode"] is not None:
            self.mode = data["mode"]
        if "next_measurement" in data and data["next_measurement"] is not None:
            self.next_measurement = data["next_measurement"]
        if "batches" in data and data["batches"] is not None:
            self.batches = data["batches"]
        if "retry_before_attempt" in data:
            self.retry_before_attempt = data["retry_before_attempt"]
        if "retry_passes" in data and data["retry_passes"] is not None:
            self.retry_passes = data["retry_passes"]
        if "stall_retries" in data and data["stall_retries"] is not None:
            self.stall_retries = data["stall_retries"]
        if "terminal" in data and data["terminal"] is not None:
            self.terminal = data["terminal"]
        if "stop_reason" in data:
            self.stop_reason = data["stop_reason"]
        if "previous_progress" in data:
            self.previous_progress = data["previous_progress"]
        if "pending" in data:
            self.pending = data["pending"]
        if "summary" in data and isinstance(data["summary"], dict):
            self.last_summary = data["summary"]

    def _load_journal(self):
        """Load journal from disk under session lock."""
        self._load_journal_read_only()

    def _apply_resume_state(self, state: Dict[str, Any]):
        """Apply verified resume_state under session lock and persist to disk."""
        self._validate_state_identity(state, source="resume_state")
        self._validate_state_fields(state)

        if "mode" in state and state["mode"] is not None:
            self.mode = state["mode"]
        if "next_measurement" in state and state["next_measurement"] is not None:
            self.next_measurement = state["next_measurement"]
        if "batches" in state and state["batches"] is not None:
            self.batches = state["batches"]
        if "retry_before_attempt" in state:
            self.retry_before_attempt = state["retry_before_attempt"]
        if "retry_passes" in state and state["retry_passes"] is not None:
            self.retry_passes = state["retry_passes"]
        if "stall_retries" in state and state["stall_retries"] is not None:
            self.stall_retries = state["stall_retries"]
        if "terminal" in state and state["terminal"] is not None:
            self.terminal = state["terminal"]
        if "stop_reason" in state:
            self.stop_reason = state["stop_reason"]
        if "previous_progress" in state:
            self.previous_progress = state["previous_progress"]
        if "pending" in state:
            self.pending = state["pending"]
        if "summary" in state and isinstance(state["summary"], dict):
            self.last_summary = state["summary"]

        self._persist_journal()

    def _adopt_backend_state(self):
        """Query backend.adopt_existing under session lock and persist to disk."""
        adopted = self.backend.adopt_existing()
        if not adopted:
            return

        # Forbid identity changes
        if "owner" in adopted and adopted["owner"] != self.owner:
            raise ValueError("Identity mismatch in adopt_existing (owner)")
        if "run_id" in adopted and adopted["run_id"] != self.run_id:
            raise ValueError("Identity mismatch in adopt_existing (run_id)")
        if "pipeline_commit" in adopted and adopted["pipeline_commit"] != self.pipeline_commit:
            raise ValueError("Identity mismatch in adopt_existing (pipeline_commit)")

        self._validate_state_fields(adopted)

        # Merge only known fields
        if "mode" in adopted and adopted["mode"] is not None:
            self.mode = adopted["mode"]
        if "next_measurement" in adopted and adopted["next_measurement"] is not None:
            self.next_measurement = adopted["next_measurement"]
        if "previous_progress" in adopted:
            self.previous_progress = adopted["previous_progress"]
        if "retry_before_attempt" in adopted:
            self.retry_before_attempt = adopted["retry_before_attempt"]
        if "retry_passes" in adopted and adopted["retry_passes"] is not None:
            self.retry_passes = adopted["retry_passes"]
        if "stall_retries" in adopted and adopted["stall_retries"] is not None:
            self.stall_retries = adopted["stall_retries"]
        if "batches" in adopted and adopted["batches"] is not None:
            self.batches = adopted["batches"]
        if "terminal" in adopted and adopted["terminal"] is not None:
            self.terminal = adopted["terminal"]
        if "stop_reason" in adopted:
            self.stop_reason = adopted["stop_reason"]
        if "pending" in adopted:
            self.pending = adopted["pending"]
        if "summary" in adopted and isinstance(adopted["summary"], dict):
            self.last_summary = adopted["summary"]

        self._persist_journal()

    def _acquire_lock(self):
        self.lock_path.parent.mkdir(parents=True, exist_ok=True)
        self._lock_file = open(self.lock_path, "a")
        try:
            fcntl.flock(self._lock_file.fileno(), fcntl.LOCK_EX | fcntl.LOCK_NB)
        except (BlockingIOError, OSError) as e:
            self._lock_file.close()
            self._lock_file = None
            raise RuntimeError(f"Could not acquire session lock on {self.lock_path}: session already running") from e

    def _release_lock(self):
        if self._lock_file is not None:
            try:
                fcntl.flock(self._lock_file.fileno(), fcntl.LOCK_UN)
            except OSError:
                pass
            try:
                self._lock_file.close()
            except OSError:
                pass
            self._lock_file = None

    def _persist_journal(self):
        self.journal_path.parent.mkdir(parents=True, exist_ok=True)
        state = {
            "owner": self.owner,
            "run_id": self.run_id,
            "pipeline_commit": self.pipeline_commit,
            "mode": self.mode,
            "next_measurement": self.next_measurement,
            "retry_passes": self.retry_passes,
            "retry_before_attempt": self.retry_before_attempt,
            "stall_retries": self.stall_retries,
            "previous_progress": self.previous_progress,
            "batches": self.batches,
            "terminal": self.terminal,
            "stop_reason": self.stop_reason,
            "pending": self.pending,
            "summary": getattr(self, "last_summary", {}),
        }
        with tempfile.NamedTemporaryFile("w", dir=str(self.journal_path.parent), delete=False, encoding="utf-8") as tf:
            json.dump(state, tf, indent=2)
            tf.flush()
            os.fsync(tf.fileno())
            temp_name = tf.name
        os.replace(temp_name, str(self.journal_path))

    def _fresh_resume_id(self) -> str:
        """Find the next unused resume-N measurement ID without overwrite."""
        start_n = 1
        if self.next_measurement and self.next_measurement.startswith("resume-"):
            try:
                start_n = int(self.next_measurement.split("-")[1]) + 1
            except (IndexError, ValueError):
                start_n = 1
        n = start_n
        while True:
            candidate = f"resume-{n:02d}"
            if not self.backend.measurement_exists(candidate):
                return candidate
            n += 1

    def _compute_continuation(self, batch: Dict[str, Any], report: Dict[str, Any]) -> Dict[str, Any]:
        new_batches = self.batches + 1
        new_mode = "resume"
        new_next_measurement = self._fresh_resume_id()
        new_retry_passes = self.retry_passes
        new_retry_before_attempt = self.retry_before_attempt
        new_stall_retries = self.stall_retries
        new_previous_progress = report.get("progress_key")
        terminal = False
        stop_reason = None

        # 1. Compute normal state decision first
        run_state = report.get("run_state")
        if run_state == "finished":
            retryable_tasks = report.get("retryable_tasks", 0)
            if retryable_tasks > 0:
                if self.retry_passes < self.max_retry_passes:
                    new_retry_passes = self.retry_passes + 1
                    new_retry_before_attempt = report.get("max_attempt_id", self.retry_before_attempt)
                    terminal = False
                    stop_reason = None
                else:
                    terminal = True
                    stop_reason = "retry_exhausted"
            else:
                terminal = True
                issues = report.get("issues", [])
                scope_complete = report.get("scope_complete", False)
                if not issues and scope_complete:
                    stop_reason = "finished"
                else:
                    stop_reason = "finished_with_issues"
        elif run_state == "paused" and report.get("pause_reason") == "references_rate_limited":
            # Preserve both retry and stall budgets; this is an external outage.
            stop_reason = "rate_limited"
        elif run_state == "paused":
            curr_progress = report.get("progress_key")
            if self.previous_progress is None or curr_progress != self.previous_progress:
                new_stall_retries = 0
                terminal = False
                stop_reason = None
            else:
                new_stall_retries = self.stall_retries + 1
                if new_stall_retries > self.max_stall_retries:
                    terminal = True
                    stop_reason = "stalled"
                else:
                    terminal = False
                    stop_reason = None
        else:
            raise ValueError(f"Invalid run_state from inspect_batch: {run_state}")

        # 2. Batch limit check if otherwise nonterminal (no batch_limit bypass after interrupted)
        if not terminal and new_batches >= self.max_batches:
            terminal = True
            stop_reason = "batch_limit"

        if (not terminal and stop_reason is None and self._call_batch_limit is not None
                and new_batches - self._call_start_batches >= self._call_batch_limit):
            stop_reason = 'session_limit'

        # 3. Overlay interrupted only if nonterminal
        if batch.get("interrupted") and not terminal:
            stop_reason = "interrupted"
            terminal = False

        return {
            "owner": self.owner,
            "run_id": self.run_id,
            "pipeline_commit": self.pipeline_commit,
            "mode": new_mode,
            "next_measurement": new_next_measurement,
            "retry_passes": new_retry_passes,
            "retry_before_attempt": new_retry_before_attempt,
            "stall_retries": new_stall_retries,
            "previous_progress": new_previous_progress,
            "batches": new_batches,
            "terminal": terminal,
            "stop_reason": stop_reason,
            "summary": report.get("summary", {}),
            "last_report": report,
        }

    def _apply_continuation(self, continuation: Dict[str, Any]):
        self.mode = continuation["mode"]
        self.next_measurement = continuation["next_measurement"]
        self.retry_passes = continuation["retry_passes"]
        self.retry_before_attempt = continuation.get("retry_before_attempt", 0)
        self.stall_retries = continuation["stall_retries"]
        self.previous_progress = continuation["previous_progress"]
        self.batches = continuation["batches"]
        self.terminal = continuation["terminal"]
        self.stop_reason = continuation["stop_reason"]
        self.last_summary = continuation.get("summary", {})
        self.last_report = continuation.get("last_report")
        self.pending = None
        self._persist_journal()

    def _result(self) -> Dict[str, Any]:
        return {
            "stop_reason": self.stop_reason,
            "terminal": self.terminal,
            "summary": getattr(self, "last_summary", {}),
            "last_report": getattr(self, "last_report", None),
            "batches": self.batches,
            "mode": self.mode,
            "next_measurement": self.next_measurement,
            "retry_passes": self.retry_passes,
            "retry_before_attempt": self.retry_before_attempt,
        }

    def run(self, *, max_batches_this_call=None) -> Dict[str, Any]:
        """Execute the automated crawl loop with atomic journal and backup verification."""
        if max_batches_this_call is not None and (type(max_batches_this_call) is not int or max_batches_this_call < 1):
            raise ValueError('max_batches_this_call phải là số nguyên dương hoặc None.')
        self._call_batch_limit = max_batches_this_call
        self._acquire_lock()
        try:
            return self._run_internal()
        finally:
            self._release_lock()

    def _run_internal(self) -> Dict[str, Any]:
        # Under lock: refresh journal from disk or initialize state
        if self.journal_path.exists():
            self._load_journal()
        elif self.resume_state is not None:
            self._apply_resume_state(self.resume_state)
        else:
            self._adopt_backend_state()

        # Saved terminal state returns without new work
        if self.terminal:
            return self._result()

        started_batches = self.batches
        self._call_start_batches = started_batches
        try:
            while not self.terminal:
                if (self.pending is None and self._call_batch_limit is not None
                        and self.batches - started_batches >= self._call_batch_limit):
                    self.stop_reason = 'session_limit'
                    self._persist_journal()
                    break
                if self.batches >= self.max_batches:
                    self.terminal = True
                    self.stop_reason = "batch_limit"
                    self._persist_journal()
                    break

                if self.pending is not None:
                    # Recovering in-flight pending batch
                    batch = self.pending["batch"]

                    if not self.pending.get("executed"):
                        if not self.backend.batch_completed(batch):
                            raise RuntimeError(
                                f"Actionable recovery error: Prepared batch {batch.get('measurement_id')} "
                                "was not completed and cannot be blindly repeated."
                            )
                        self.pending["executed"] = True
                        self._persist_journal()

                    # If pending has after candidate, preserve candidate exactly during backup retry
                    if self.pending.get("after") is not None:
                        after = self.pending["after"]
                        report = self.pending.get("report")
                        if report is None:
                            report = self.backend.inspect_batch(batch)
                            self.pending["report"] = report
                            self._persist_journal()
                    else:
                        report = self.backend.inspect_batch(batch)
                        after = self._compute_continuation(batch, report)
                        self.pending["after"] = after
                        self.pending["report"] = report
                        self._persist_journal()

                    self.backend.backup_batch(batch, report, after)
                    self._apply_continuation(after)

                    print(
                        f"[AutoCrawlSession] Recovered batch {batch.get('measurement_id')} verified & backed up. "
                        f"State={report.get('run_state')}, stop_reason={self.stop_reason or 'continuing'}"
                    )

                    if self.stop_reason in ("interrupted", "rate_limited", "session_limit") or self.terminal:
                        break

                else:
                    # Preflight errors leave no unexecuted pending batch behind.
                    preflight = getattr(self.backend, 'preflight', None)
                    if preflight is not None:
                        preflight()
                    # Fresh batch execution
                    batch = self.backend.make_batch(self.mode, self.next_measurement,
                                                    retry_before_attempt=self.retry_before_attempt)
                    print(
                        f"[AutoCrawlSession] Batch {self.batches + 1}: "
                        f"mode={self.mode}, measurement={self.next_measurement}"
                    )

                    self.pending = {
                        "batch": batch,
                        "executed": False,
                        "after": None,
                    }
                    self._persist_journal()

                    batch = self.backend.execute_batch(batch)
                    self.pending["batch"] = batch
                    self.pending["executed"] = True
                    self._persist_journal()

                    report = self.backend.inspect_batch(batch)
                    after = self._compute_continuation(batch, report)
                    self.pending["after"] = after
                    self.pending["report"] = report
                    self._persist_journal()

                    self.backend.backup_batch(batch, report, after)
                    self._apply_continuation(after)

                    print(
                        f"[AutoCrawlSession] Batch {self.batches} verified & backed up. "
                        f"State={report.get('run_state')}, stop_reason={self.stop_reason or 'continuing'}"
                    )

                    if self.stop_reason in ("interrupted", "rate_limited", "session_limit") or self.terminal:
                        break

        except KeyboardInterrupt:
            # User policy: best-effort backup of a measured/completed batch then stop
            if self.pending is not None and (
                self.pending.get("executed")
                or self.backend.batch_completed(self.pending["batch"])
            ):
                batch = self.pending["batch"]
                self.pending["executed"] = True

                if self.pending.get("after") is not None:
                    # Preserve precomputed continuation exactly except stop_reason=interrupted for nonterminal;
                    # do not increment batch/retry counters twice.
                    # Preserve terminal finished/retry_exhausted/stalled states rather than reopening.
                    after = dict(self.pending["after"])
                    report = self.pending.get("report")
                    if report is None:
                        report = self.backend.inspect_batch(batch)
                        self.pending["report"] = report
                    if not after.get("terminal"):
                        after["stop_reason"] = "interrupted"
                    self.pending["after"] = after
                    self._persist_journal()
                else:
                    # If continuation not computed yet, compute normal state decision first then overlay interrupted stop;
                    # finished+retryable must still count the planned first retry so restarting cannot gain a third retry pass.
                    # Preserve terminal finished/retry_exhausted/stalled states rather than reopening.
                    report = self.pending.get("report")
                    if report is None:
                        report = self.backend.inspect_batch(batch)
                        self.pending["report"] = report
                    after = self._compute_continuation(batch, report)
                    if not after.get("terminal"):
                        after["stop_reason"] = "interrupted"
                    self.pending["after"] = after
                    self._persist_journal()

                print(
                    f"[AutoCrawlSession] Ctrl+C caught: performing best-effort backup for batch {batch.get('measurement_id')}"
                )
                # Repeat interrupt during backup may propagate:
                self.backend.backup_batch(batch, report, after)
                self._apply_continuation(after)
                print(
                    f"[AutoCrawlSession] Interrupted batch {batch.get('measurement_id')} verified & backed up. Stop reason={self.stop_reason}"
                )
            else:
                raise

        if self.terminal:
            print(
                f"[AutoCrawlSession] Session finished: terminal=True, reason={self.stop_reason}, batches={self.batches}"
            )
        elif self.stop_reason == "interrupted":
            print(f"[AutoCrawlSession] Session stopped: interrupted (batches={self.batches})")

        return self._result()


## 3. Xác nhận thư mục Drive chung

Đã cấu hình [thư mục chung của nhóm](https://drive.google.com/drive/folders/1w9cVHms6Q9aakHKMSJY5DGzmBRAPleeT?usp=drive_link), ID `1w9cVHms6Q9aakHKMSJY5DGzmBRAPleeT`. Cell dưới yêu cầu cấp quyền mount, ưu tiên đường dẫn lối tắt theo ID khi Drive hiển thị nó, sau đó dùng `TEAM_DRIVE_DIR` nếu đường dẫn đó đã có. Không dùng URL làm đường dẫn file.

Nếu thư mục chưa hiện trong Drive đã mount, mở link trên bằng đúng tài khoản Colab và thêm [lối tắt vào Drive của tôi](https://support.google.com/drive/answer/9700156), rồi chạy lại cell. Nếu là chủ thư mục, có thể sao chép đường dẫn thực tế từ bảng Files vào `TEAM_DRIVE_DIR`. Không tạo thư mục riêng khác cùng tên; lối tắt không tự cấp quyền truy cập.

Auto-run cần Drive kể cả pilot. Corpus/SQLite luôn ở ổ VM; chỉ ghi snapshot và số liệu vào thư mục chung. Cell dừng nếu chưa tìm thấy thư mục hoặc chưa có quyền ghi.


In [ ]:
TEAM_PATH = None
from google.colab import drive
drive.mount("/content/drive")
TEAM_PATH = resolve_team_drive(TEAM_DRIVE_FOLDER_URL, TEAM_DRIVE_DIR)
with tempfile.NamedTemporaryFile(dir=TEAM_PATH, prefix=f"access-test-{OWNER}-", delete=True) as probe:
    probe.write(b"ok")
    probe.flush()
TEAM_DRIVE_DIR = str(TEAM_PATH)
print("Link thư mục chung:", TEAM_DRIVE_FOLDER_URL)
print("Đã kiểm tra quyền ghi:", TEAM_PATH)


## 4. Khôi phục nếu cần, rồi chạy tự động

Lần đầu để `RESTORE_BUNDLE` rỗng. Trong VM mới, chọn `resume`, giữ owner/run ID/commit, điền đường dẫn ZIP mới nhất của run này trên Drive và giữ `CORPUS_PATH` rỗng.

Trong cùng VM, journal và con trỏ corpus tự giữ tiến độ. Nếu đã dừng do lỗi backup, sửa lỗi rồi Run all để hoàn tất backup trước khi crawl tiếp. Không đổi run ID để né lỗi.


### Khôi phục từ gói bàn giao (tùy chọn)

Cell dưới kiểm tra checksum, đọc đúng snapshot trong ZIP và restore bằng CLI; không sửa ZIP gốc. Gói auto chứa continuation để giữ số lượt retry đã dùng. Run all trong cùng VM tiếp tục corpus hiện tại thay vì nạp lại ZIP cũ.

Khi nâng code từ snapshot cũ: dừng phiên cũ sau sao lưu, dùng WORK_ROOT mới; điền SHA cũ vào `PREVIOUS_PIPELINE_COMMIT`, SHA mới vào `COMMIT_SHA`, giữ owner/run ID. Dữ liệu và provenance cũ được giữ nguyên, có receipt nâng code. Lượt retry cũ thiếu mốc duyệt sẽ dừng để kiểm tra.


In [ ]:
# A local pointer keeps the current restored corpus across Run all in the same VM.
pointer_path = WORK_ROOT / "automation" / RUN_ID / "corpus.json"
current_corpus = None
if pointer_path.exists():
    pointer = json.loads(pointer_path.read_text())
    if (pointer.get("owner"), pointer.get("run_id"), pointer.get("pipeline_commit")) != (OWNER, RUN_ID, COMMIT_SHA):
        raise ValueError("Con trỏ corpus không khớp owner/run ID/commit.")
    candidate = Path(pointer["output_root"])
    if (candidate / "state.sqlite").is_file():
        current_corpus = candidate
        RESTORED_AUTOMATION = pointer.get("automation")

if RESTORE_BUNDLE and current_corpus is not None:
    CORPUS_PATH = str(current_corpus)
    print("Tiếp tục corpus hiện tại, không nạp lại ZIP cũ:", CORPUS_PATH)
elif RESTORE_BUNDLE:
    import zipfile
    if RUN_MODE != "resume" or TEAM_PATH is None:
        raise ValueError("Khôi phục cần RUN_MODE=resume và Drive đã được mount.")
    bundle_source = Path(RESTORE_BUNDLE)
    checksum_source = bundle_source.with_suffix(".zip.sha256")
    expected = checksum_source.read_text().split()[0]
    if not re.fullmatch(r"[0-9a-f]{64}", expected):
        raise ValueError("Checksum bàn giao không đúng định dạng.")
    restore_dir = WORK_ROOT / "restored" / RUN_ID / MEASUREMENT_ID
    if restore_dir.exists():
        raise FileExistsError("Đích restore đã có; không ghi đè corpus cũ.")
    incoming = WORK_ROOT / "restore-input" / RUN_ID / MEASUREMENT_ID
    incoming.mkdir(parents=True, exist_ok=True)
    local_bundle = copy_verified(bundle_source, incoming / bundle_source.name, expected_sha256=expected)
    with zipfile.ZipFile(local_bundle) as archive:
        identity = json.loads(archive.read("run.json"))
        previous_commit = globals().get("PREVIOUS_PIPELINE_COMMIT", "")
        if previous_commit and not re.fullmatch(r"[0-9a-f]{40}", previous_commit):
            raise ValueError("PREVIOUS_PIPELINE_COMMIT phải là SHA cũ 40 ký tự.")
        source_commit = identity.get("pipeline_commit")
        if (identity["owner"] != OWNER or identity["run_id"] != RUN_ID
                or identity["is_mock"] is not False
                or not isinstance(source_commit, str) or not re.fullmatch(r"[0-9a-f]{40}", source_commit)
                or source_commit not in {COMMIT_SHA, previous_commit}):
            raise ValueError("Gói không khớp owner/run ID/commit hoặc là mock.")
        restored_policy = identity.get("automation")
        if source_commit != COMMIT_SHA:
            if (not isinstance(restored_policy, dict)
                    or (restored_policy.get("owner"), restored_policy.get("run_id"), restored_policy.get("pipeline_commit"))
                       != (OWNER, RUN_ID, source_commit)):
                raise ValueError("Gói thiếu continuation hợp lệ để nâng code; giữ nguyên snapshot.")
            if restored_policy.get("retry_passes", 0) and "retry_before_attempt" not in restored_policy:
                raise ValueError("Lượt retry cũ thiếu mốc duyệt; cần kiểm tra trước khi nâng code.")
            restored_policy = {**restored_policy, "pipeline_commit": COMMIT_SHA,
                               "retry_before_attempt": restored_policy.get("retry_before_attempt", 0)}
        snapshot = incoming / "snapshot.tar.gz"
        with archive.open("snapshot.tar.gz") as source, snapshot.open("wb") as destination:
            shutil.copyfileobj(source, destination, length=1024 * 1024)
    run_snapshot_cli(["verify", "--archive", str(snapshot)])
    run_snapshot_cli(["restore", "--archive", str(snapshot), "--destination", str(restore_dir)])
    CORPUS_PATH = str(restore_dir / "corpus")
    RESTORED_AUTOMATION = restored_policy
    if source_commit != COMMIT_SHA:
        automation_json(restore_dir / "pipeline-upgrade.json", {"from_commit": source_commit,
            "to_commit": COMMIT_SHA, "source_bundle_sha256": expected})
        print("Đã nâng code từ snapshot đã kiểm tra; giữ nguyên dữ liệu và provenance cũ.")
    automation_json(pointer_path, {"owner": OWNER, "run_id": RUN_ID,
        "pipeline_commit": COMMIT_SHA, "output_root": CORPUS_PATH, "automation": RESTORED_AUTOMATION})
    print("Corpus đã restore:", CORPUS_PATH)


### Chạy và đo theo đợt

Vòng tự động ở cell cuối chạy runner theo từng đợt, giữ riêng cold và resume. Không cần tự đổi measurement sau mỗi đợt.

Trước mỗi batch dữ liệu thật, notebook thử một request references với key và `limit=1`. Phép thử nằm ngoài benchmark; khi có corpus cũ, slot request/cooldown được ghi qua scheduler chung. Nếu probe lỗi, batch mới chưa được tạo. `rate_limited` nghĩa là đã sao lưu sau chuỗi 429 rồi dừng; `session_limit` nghĩa là đã sao lưu đủ số batch thử cho lần bấm chạy này. Cả hai chưa phải hoàn tất crawl.


## 5. Kiểm tra sau mỗi đợt

Vòng điều phối kiểm tra cấu trúc và xuất statistics trước khi backup. `validation.valid=true` chỉ xác nhận dữ liệu nhất quán; `quality.valid=false` phải đọc từng issue. PDF-only/withdrawn giữ nguyên trạng thái thiếu, không đổi thành success.

Benchmark đo từng invocation của runner, không gồm kiểm tra/đóng gói/upload. Resume chỉ đo task mới; discovery không chạy lại là null. Thời gian từng bài là khoảng elapsed, có thể chồng nhau; không cộng thành wall time toàn dải.


### Kiểm tra sau mỗi đợt

Inspector và statistics chạy trước khi sao lưu. Lỗi cấu trúc dừng phiên; PDF-only/withdrawn được giữ nguyên và báo trong phần thiếu.


## 6. Sao lưu rồi mới chạy đợt tiếp

Đầu ra: `<Drive chung>/<owner>/<run_id>/<measurement_id>/`. Mỗi gói ZIP gồm snapshot, benchmark/timing, statistics và run.json. Chương trình verify snapshot, đối chiếu SHA-256 bản Drive rồi mới tiếp tục. Không ghi đè file khác nội dung; backup lỗi làm dừng.

Theo lựa chọn chung: giữ **2 snapshot ZIP/checksum mới nhất của run**, giữ **mọi JSON benchmark, timing, statistics, run.json** riêng trên Drive. Chỉ dọn bản cũ sau khi hai bản giữ lại được kiểm tra; corpus/JSON local không bị xóa. Cần đủ dung lượng cho bản mới trước khi dọn.

Khi Stop/Ctrl+C: cố gắng sao lưu đợt đã đo rồi dừng. Ngắt lần hai hoặc mất VM đột ngột có thể làm mất phần sau snapshot cuối. Không có backup nền khi runner còn chạy.


### Sao lưu trước khi tiếp tục

Sau mỗi đợt, chương trình export/verify snapshot, đóng gói số liệu và đối chiếu SHA-256 trên Drive. Chỉ sau đó nó mới bắt đầu đợt tiếp theo. Lỗi Drive làm dừng; chạy lại cùng notebook sẽ thử hoàn tất backup đang chờ, không crawl lại đợt đã đo.


In [ ]:
# Run this cell once; each batch is verified and backed up before the next starts.
ensure_release()
if TEAM_PATH is None:
    raise ValueError("Auto-run cần Drive đã mount và kiểm tra quyền ghi.")
if CORPUS_PATH:
    selected_root = Path(CORPUS_PATH)
elif current_corpus is not None:
    selected_root = current_corpus
else:
    selected_root = WORK_ROOT / RUN_ID
if selected_root.resolve().is_relative_to(Path("/content/drive").resolve()):
    raise ValueError("Corpus/SQLite phải ở ổ VM, không đặt trong Drive.")
automation_json(pointer_path, {"owner": OWNER, "run_id": RUN_ID,
    "pipeline_commit": COMMIT_SHA, "output_root": str(selected_root), "automation": RESTORED_AUTOMATION})
initial_args = ["--owner", OWNER]
initial_args += ["--full-range"] if RUN_MODE == "full" else ["--max-candidates", str(MAX_CANDIDATES), "--max-papers", str(MAX_PAPERS)]
worker = NotebookBatchWorker(repo_dir=REPO_DIR, env=PIPELINE_ENV, work_root=WORK_ROOT,
    output_root=selected_root, drive_root=TEAM_PATH, owner=OWNER, run_id=RUN_ID,
    pipeline_commit=COMMIT_SHA, initial_args=initial_args, stop_after_tasks=STOP_AFTER_TASKS,
    max_files=SNAPSHOT_MAX_FILES, cli=run_cli, stop_after_seconds=STOP_AFTER_SECONDS,
    references_rate_limit_threshold=REFERENCES_RATE_LIMIT_THRESHOLD, require_s2_key=REQUIRE_S2_KEY, restored_state=RESTORED_AUTOMATION,
    historical_pipeline_commit=globals().get("PREVIOUS_PIPELINE_COMMIT") or None)
session = AutoCrawlSession(worker, owner=OWNER, run_id=RUN_ID, pipeline_commit=COMMIT_SHA,
    journal_path=WORK_ROOT / "automation" / RUN_ID / "session.json",
    initial_mode=RUN_MODE, initial_measurement=MEASUREMENT_ID,
    max_retry_passes=AUTO_RETRY_PASSES, max_batches=AUTO_MAX_BATCHES,
    resume_state=RESTORED_AUTOMATION)
result = session.run(max_batches_this_call=AUTO_BATCHES_PER_CALL)
print(json.dumps({k: result.get(k) for k in ("stop_reason", "terminal", "batches", "retry_passes", "summary")}, ensure_ascii=False, indent=2))


## 7. Đọc kết quả và tiếp tục

Cuối cell tự động có `stop_reason`, `terminal`, số batch/retry và tổng kết. `finished` là hoàn tất theo scope; `finished_with_issues` còn ngoại lệ hoặc scope pilot bị giới hạn; `retry_exhausted` đã hết hai lượt retry. `stalled`/`batch_limit` cần kiểm tra trước khi chạy tiếp. `terminal=true` không tự chứng minh mọi bài đều có TeX.

Sau lỗi Drive, sửa lỗi rồi chạy lại cùng cấu hình trong VM; notebook hoàn tất pending backup trước. Khi mất VM, chọn `resume`, điền `RESTORE_BUNDLE` là ZIP mới nhất còn trên Drive, giữ owner/run ID/commit rồi Run all và mount lại. Không tự sửa SQLite hoặc đặt lại số retry. Snapshot cuối là mốc phục hồi; dữ liệu chưa sao lưu có thể mất.

Leader nhận JSON của mọi measurement và một snapshot đã kiểm tra cho mỗi owner. Xem [README](https://github.com/ChickenDev-cyber/arxiv-data-science#chay-chinh-thuc) để ghép collection và chuyển số liệu sang `arvix-report`. Số đo theo đợt phải ghi đúng phạm vi; không gọi một batch là thời gian toàn dải, không cộng thời gian năm máy thành wall time chung.

Test offline không thay thế việc thử upload/restore trên tài khoản Drive thực tế. Không thay notebook đang chạy; dùng bản cập nhật sau khi phiên cũ đã dừng và sao lưu.
